In [14]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

from xgboost import XGBClassifier

In [3]:
df = pd.read_csv("datasets/improved_credit_risk.csv")

In [4]:
df.head(10)

,Age,Annual_Income,Monthly_Inhand_Salary,Num_Bank_Accounts,Num_Credit_Card,Interest_Rate,Num_of_Loan,Delay_from_due_date,Num_of_Delayed_Payment,Changed_Credit_Limit,...,normalized_utilization_risk,normalized_inquiry_intensity,normalized_investment_ratio,normalized_loan_burden_index,behavioral_risk_indicator,credit_mix_quality,normalized_savings_capacity_ratio,population_density_factor,Default_Flag,Borrower_Tier
0,28,34847.840,3037.986667,2,4,6,1,3,1,5.42,...,1.0,1.000000,0.013058,1.0,0,2,0.980744,0.96,0,Prime
1,28,34847.840,3037.986667,2,4,6,1,3,3,5.42,...,1.0,1.000000,0.082800,1.0,0,2,0.910979,1.04,1,Prime
2,55,30689.890,2612.490833,2,5,4,1,5,6,1.99,...,1.0,0.666667,0.073421,1.0,0,2,0.920267,0.96,0,Near_Prime
3,55,4148862.000,2612.490833,2,5,4,1,6,6,1.99,...,1.0,1.000000,0.089510,1.0,0,2,0.904172,0.96,1,Near_Prime
4,45,31370.800,2825.233333,1,6,12,2,2,3,5.76,...,1.0,0.714286,0.038716,1.0,0,2,0.944770,0.96,0,Prime
5,36,54392.160,4766.680000,6,4,14,3,10,8,5.54,...,1.0,1.000000,0.069122,1.0,1,1,0.904767,1.04,1,Prime
6,36,54392.160,4766.680000,6,4,14,3,10,10,5.54,...,1.0,1.000000,0.030482,1.0,1,1,0.943415,0.96,0,Prime
7,39,8701.545,519.128750,6,5,32,7,23,10,8.86,...,1.0,1.000000,0.072379,1.0,1,1,0.857079,1.08,1,Prime
8,37,25546.260,2415.855000,8,7,14,5,16,15,1.83,...,1.0,0.875000,0.058361,1.0,1,1,0.899672,1.04,0,Prime
9,31,31993.780,2942.148333,6,6,7,2,8,14,6.28,...,1.0,0.142857,0.029232,1.0,1,1,0.955415,1.04,1,Near_Prime


In [5]:
df.shape

(8744, 36)

In [6]:
df.replace([np.inf, -np.inf], np.nan, inplace=True)
df.fillna(df.median(numeric_only=True), inplace=True)

In [7]:
encoder = LabelEncoder()

df["Payment_of_Min_Amount"] = encoder.fit_transform(df["Payment_of_Min_Amount"])

In [8]:
df.shape

(8744, 36)

In [9]:
df = pd.get_dummies(df, columns=["Credit_Mix", "Payment_Behaviour", "Borrower_Tier"], drop_first=True)

In [10]:
df.shape

(8744, 42)

In [11]:
X = df.drop("Default_Flag", axis=1)
y = df["Default_Flag"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

In [12]:
scale_pos_weight = len(y_train[y_train==0]) / len(y_train[y_train==1])

In [13]:
model = XGBClassifier(
    objective="binary:logistic",
    eval_metric="auc",
    scale_pos_weight=scale_pos_weight,
    n_estimators=300,
    learning_rate=0.05,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

model.fit(X_train, y_train)

,objective,'binary:logistic'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,0.8
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,'auc'


In [15]:
y_pred_proba = model.predict_proba(X_test)[:,1]

roc_auc_score(y_test, y_pred_proba)

0.6125863129414117